# tf.function playground — trace it, break it, debug it

Companion notebook to **[Array Frameworks, Chapter 2: TensorFlow](https://an-bluecat.github.io/ml-courses/frameworks_course/2_tensorflow.html)**.

Everything from the chapter's tracing pages, live. Run cells top to bottom (`Shift+Enter`), then edit anything and re-run — that's the point.

In [ ]:
import tensorflow as tf
print("TF", tf.__version__)

## 1. The print experiment — Python runs once, the graph runs every time

Predict before you run: which line prints on call 1? Which on call 2?

In [ ]:
@tf.function
def f(x):
    print("  [python print] tracing now, x =", x)   # python side effect
    tf.print("  [tf.print] running, x =", x)         # a graph NODE
    return x * 2.0

print("call 1:", f(tf.constant(3.0)).numpy())
print("call 2:", f(tf.constant(4.0)).numpy())

**Things to try:**
- Call `f(tf.constant([1.0, 2.0]))` — new *shape*, so it traces again. Watch the python print fire once more.
- Call `f(3.0)` with a plain Python float a few times with different values — retrace per value (next section).
- Comment out `@tf.function` and re-run — now both prints fire every call: you're back in eager land.

## 2. Retracing — Python values bake into the graph

In [ ]:
@tf.function
def g(x):
    print("  [tracing] with x =", x)
    return x + 1

g(tf.constant(1.0)); g(tf.constant(2.0))   # tensors, same signature: ONE trace
g(1); g(2); g(3)                           # python ints: a NEW trace each value!
print("traces so far:", len(g.experimental_get_tracing_count() if callable(getattr(g, 'experimental_get_tracing_count', None)) else []) or g.experimental_get_tracing_count())

## 3. AutoGraph — see the code TF generates from your `if`

In [ ]:
def h(x):
    if x > 0:
        return x * 2.0
    else:
        return -x

hf = tf.function(h)
print("h(2.0)  ->", hf(tf.constant(2.0)).numpy())
print("h(-2.0) ->", hf(tf.constant(-2.0)).numpy())
print()
print(tf.autograph.to_code(h))   # the rewritten source, for real

## 4. The debugger question — how do you actually step through a tf.function?

You can't put a breakpoint *inside a graph* — by run time your Python is gone (that's the whole chapter). The real-world technique: **temporarily turn tracing off** so the function runs as plain eager Python, debug it there, then turn it back on.

In [ ]:
tf.config.run_functions_eagerly(True)    # every @tf.function now runs as plain Python

@tf.function
def buggy(x):
    y = x * 2.0
    # breakpoint()   # <- uncomment: a REAL pdb prompt, because this is now real Python
    print("  y is a real value I can inspect:", y.numpy())
    return y - 10.0

print(buggy(tf.constant(3.0)).numpy())

tf.config.run_functions_eagerly(False)   # back to traced/compiled behavior
print("eager mode off again")

Uncomment the `breakpoint()` line and re-run: you get an interactive pdb session (type `p y`, `n`, `c`). This works because `run_functions_eagerly(True)` makes TF skip tracing entirely — the identical trick exists in the other frameworks:

| Framework | "let me debug it" switch |
|---|---|
| TF | `tf.config.run_functions_eagerly(True)` |
| JAX | `with jax.disable_jit():` |
| PyTorch | (already eager — just don't call `torch.compile`) |

And for logging *inside* real traced runs without disabling anything: `tf.print` (here) / `jax.debug.print` (Chapter 10) — ops compiled into the graph itself.

## 5. Free experiments

1. Write `k(x)` with a Python `for _ in range(3)` loop inside `@tf.function` — how many ops does the loop become? (`print(tf.function(k).get_concrete_function(tf.TensorSpec([], tf.float32)).graph.as_graph_def())` — count the `mul` nodes.)
2. Make `g` above take `n` steps as a Python int and watch the retrace counter climb — then fix it by passing `tf.constant(n)` instead.
3. Put `x.numpy()` inside a `@tf.function` and call it — read the error: that's the tracer refusing to produce a concrete value, TF's cousin of JAX's `TracerBoolConversionError` from Chapter 4.